## Neuron Extraction

This notebook is to extract a neuron out of multiple ZIP files.

In [53]:
from zipfile import ZipFile
import re
import os
import io

from caveclient import CAVEclient
client = CAVEclient()

In [42]:
file_sources = [
                # Sven L2 graphs are in a different format
                # "/Users/eric/nobackup/fafb/630/l2_skeletons/skeleton_cache_630.zip",
                "/Users/eric/nobackup/fafb/630/skeletons/sk_lod1_630_healed.zip",
                "/Users/eric/nobackup/fafb/630/skeletons/sk_lod1_630_healed_ds16.zip",
                "/Users/eric/nobackup/fafb/630/skeletons/sk_lod1_630_healed_ds2.zip",
                "/Users/eric/nobackup/fafb/630/skeletons/sk_lod1_630_healed_ds4.zip",
                "/Users/eric/nobackup/fafb/630/skeletons/sk_lod1_630_healed_ds8.zip",
                "/Users/eric/nobackup/fafb/630/skeletons/sk_lod1_630_raw.zip",
               ]
#file_sources = file_sources[0:2]

In [54]:
neurons_to_fetch = set()
# neurons_to_fetch = set([720575940630513208, 720575940619136414, 720575940621198017])

In [69]:
davi_neurons = set([720575940610938902, 720575940611114262, 720575940611494629, 720575940612761854,
                    720575940614619458, 720575940614630486, 720575940615904598, 720575940618617536, 
                    720575940622150663, 720575940623959344, 720575940626931112, 720575940628719355,
                    720575940629684792, 720575940633744791, 720575940637503310, 720575940637836477, 
                    720575940639433424])

datastack_name = 'flywire_fafb_production'
client = CAVEclient(datastack_name)
timestamp = client.materialize.get_version_metadata(630)['time_stamp']

for neuron_id in davi_neurons:
    # Get supervoxels
    supervoxels = client.chunkedgraph.get_leaves(neuron_id)
    #print(supervoxels)
    new_id = client.chunkedgraph.get_root_id(supervoxel_id=supervoxels[0], timestamp=timestamp)
    print(neuron_id, new_id)

    neurons_to_fetch.add(new_id)

720575940618617536 720575940618617536
720575940633744791 720575940633744791
720575940614619458 720575940644122903
720575940611494629 720575940611494629
720575940622150663 720575940624884112
720575940626931112 720575940611596517
720575940629684792 720575940629684792
720575940637503310 720575940631652433
720575940623959344 720575940623501129
720575940639433424 720575940620030555
720575940610938902 720575940610938902
720575940611114262 720575940611114262
720575940614630486 720575940618514555
720575940615904598 720575940623997879
720575940628719355 720575940628719355
720575940637836477 720575940637836477
720575940612761854 720575940612761854


In [70]:
out_base = "swc_out"

In [71]:
# Iterate through each SWC in each zip file
for filename in file_sources:
    short_filename = os.path.splitext(os.path.basename(filename))[0]
    out_dir = os.path.join(out_base, short_filename)
    os.makedirs(out_dir, exist_ok=True)
    with ZipFile(filename, 'r') as myzip:
        for zipinfo in myzip.infolist():
            neuron_file_match = re.match('([0-9]+)\.swc', zipinfo.filename)
            if not neuron_file_match:
                continue
            neuron_id = int(neuron_file_match.group(1))
            if neuron_id in neurons_to_fetch:
                swc_data = io.BytesIO(myzip.read(zipinfo)).read()
                swc_path = os.path.join(out_dir, f'{neuron_id}.swc')
                print(swc_path)
                with open(swc_path, "wb") as f:
                    f.write(swc_data)

swc_out/sk_lod1_630_healed/720575940610938902.swc
swc_out/sk_lod1_630_healed/720575940618514555.swc
swc_out/sk_lod1_630_healed/720575940629684792.swc
swc_out/sk_lod1_630_healed/720575940624884112.swc
swc_out/sk_lod1_630_healed/720575940612761854.swc
swc_out/sk_lod1_630_healed/720575940611494629.swc
swc_out/sk_lod1_630_healed/720575940631652433.swc
swc_out/sk_lod1_630_healed/720575940623997879.swc
swc_out/sk_lod1_630_healed/720575940637836477.swc
swc_out/sk_lod1_630_healed/720575940623501129.swc
swc_out/sk_lod1_630_healed/720575940620030555.swc
swc_out/sk_lod1_630_healed/720575940611114262.swc
swc_out/sk_lod1_630_healed/720575940628719355.swc
swc_out/sk_lod1_630_healed/720575940633744791.swc
swc_out/sk_lod1_630_healed/720575940618617536.swc
swc_out/sk_lod1_630_healed/720575940611596517.swc
swc_out/sk_lod1_630_healed/720575940644122903.swc
swc_out/sk_lod1_630_healed_ds16/720575940610938902.swc
swc_out/sk_lod1_630_healed_ds16/720575940618514555.swc
swc_out/sk_lod1_630_healed_ds16/72057594

In [72]:
neurons_to_fetch

{720575940610938902,
 720575940611114262,
 720575940611494629,
 720575940611596517,
 720575940612761854,
 720575940618514555,
 720575940618617536,
 720575940620030555,
 720575940623501129,
 720575940623997879,
 720575940624884112,
 720575940628719355,
 720575940629684792,
 720575940631652433,
 720575940633744791,
 720575940637836477,
 720575940644122903}